# 01 · Data collection and understanding

Question: what does the fictional source contain, and is it suitable for aggregate descriptive analysis? No cleaning happens in this notebook.

Source: [IBM HR Analytics Employee Attrition & Performance](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset), published by Pavan Subhash. Kaggle lists Database: Open Database, Contents: Database Contents. Full attribution and licence links: `data/SOURCE.md`. The dataset does not represent IBM’s actual workforce.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'retainwise').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from retainwise.data import RAW, PROCESSED, DATA, validate_source, prepare, dictionary, write_version, load_data
from retainwise.analysis import rate_table, distribution, hypothesis_evidence, HYPOTHESES
pd.set_option('display.max_columns', 40)


In [2]:
raw = pd.read_csv(RAW)
report = validate_source(raw)
display(pd.Series(report, name='Validation evidence'))
print('Shape:', raw.shape)

rows                                                   1470
columns                                                  35
missing_values                                            0
duplicate_rows                                            0
identifier_unique                                      True
constant_columns     [EmployeeCount, Over18, StandardHours]
target_counts                      {'No': 1233, 'Yes': 237}
Name: Validation evidence, dtype: object

Shape: (1470, 35)


## Schema, values and categories

Inspect every field without exposing employee-level rows in the dashboard. This notebook is a development artefact using fictional data.

In [3]:
profile = pd.DataFrame({'dtype': raw.dtypes.astype(str), 'missing': raw.isna().sum(), 'unique': raw.nunique()})
display(profile)
for col in raw:
    if raw[col].nunique() <= 10:
        print(col, sorted(raw[col].unique().tolist()))
print('Duplicates:', raw.duplicated().sum())
print('Identifier unique:', raw.EmployeeNumber.is_unique)
print('Constant columns:', [c for c in raw if raw[c].nunique() == 1])

,dtype,missing,unique
Age,int64,0,43
Attrition,object,0,2
BusinessTravel,object,0,3
DailyRate,int64,0,886
Department,object,0,3
DistanceFromHome,int64,0,29
Education,int64,0,5
EducationField,object,0,6
EmployeeCount,int64,0,1
EmployeeNumber,int64,0,1470


Attrition ['No', 'Yes']
BusinessTravel ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']
Department ['Human Resources', 'Research & Development', 'Sales']
Education [1, 2, 3, 4, 5]
EducationField ['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']
EmployeeCount [1]
EnvironmentSatisfaction [1, 2, 3, 4]
Gender ['Female', 'Male']
JobInvolvement [1, 2, 3, 4]
JobLevel [1, 2, 3, 4, 5]
JobRole ['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative']
JobSatisfaction [1, 2, 3, 4]
MaritalStatus ['Divorced', 'Married', 'Single']
NumCompaniesWorked [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Over18 ['Y']
OverTime ['No', 'Yes']
PerformanceRating [3, 4]
RelationshipSatisfaction [1, 2, 3, 4]
StandardHours [80]
StockOptionLevel [0, 1, 2, 3]
TrainingTimesLastYear [0, 1, 2, 3, 4, 5, 6]
WorkLifeBalance [1, 2, 3, 4]
Duplicates: 0
Identifier

In [4]:
display(raw.Attrition.value_counts().rename_axis('Attrition').to_frame('n').assign(share_pct=lambda x: 100*x.n/len(raw)))
display(dictionary(raw.drop(columns=['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours'])))

,n,share_pct
Attrition,,
No,1233,83.877551
Yes,237,16.122449


,field,dtype,description,unique_values,missing
0,Age,int64,Age in years; used only for aggregate audit.,43,0
1,Attrition,object,Source outcome: Yes / No; no exit date is supp...,2,0
2,BusinessTravel,object,Source travel-frequency category; no measured ...,3,0
3,DailyRate,int64,Source daily rate; units and interpretation no...,886,0
4,Department,object,Source organisational department.,3,0
5,DistanceFromHome,int64,Source distance from home; distance unit not d...,29,0
6,Education,int64,Source ordinal scale: 1=Below college; 2=Colle...,5,0
7,EducationField,object,Source education field category.,6,0
8,EnvironmentSatisfaction,int64,Source ordinal scale: 1=Low; 2=Medium; 3=High;...,4,0
9,Gender,object,Source-labelled Female/Male; incomplete repres...,2,0


## Suitability and limits

The source passes the explicit v1 validation checks. The target is imbalanced, there is no dated observation window and currency is not documented. Department, role and other variables may confound crude comparisons. The binary Gender field is incomplete representation. Analysis can describe associations within this fictional population; it cannot explain departures, estimate policy effects or generalise to an employer. The identifier and constants will be removed only in notebook 02, after this evidence check.